In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from collections import defaultdict

SEED = 42
np.random.seed(SEED)

# Load MTA dataset
df = pd.read_csv('../data/processed/df_mta.csv', parse_dates=['touchpoint_date'])

print(df.shape)
print(df.head())
print(f"\nChannels: {df['channel'].unique()}")
print(f"Total users: {df['user_id'].nunique()}")
print(f"Converted users: {df[df['conversion']==1]['user_id'].nunique()}")

(30129, 4)
   user_id touchpoint_date      channel  conversion
0        0      2023-03-16  paid_search           0
1        0      2023-03-17      organic           0
2        1      2022-08-15      display           0
3        1      2022-08-18      display           0
4        1      2022-08-18  paid_search           0

Channels: <ArrowStringArray>
['paid_search', 'organic', 'display', 'paid_social', 'affiliate']
Length: 5, dtype: str
Total users: 10000
Converted users: 3005


In [2]:
# ── Build user journeys ───────────────────────────────────────────────────────

# Group touchpoints into paths per user
journeys = (
    df.sort_values(['user_id', 'touchpoint_date'])
      .groupby('user_id')
      .agg(
          path=('channel', list),
          converted=('conversion', 'max')
      )
      .reset_index()
)

print(journeys.shape)
print(journeys.head(10))
print(f"\nConverted journeys: {journeys['converted'].sum()}")
print(f"Non-converted journeys: {(journeys['converted'] == 0).sum()}")

(10000, 3)
   user_id                                               path  converted
0        0                             [paid_search, organic]          0
1        1       [display, display, paid_search, paid_social]          0
2        2  [paid_search, organic, paid_social, affiliate,...          0
3        3                                          [display]          1
4        4  [display, organic, paid_search, paid_search, d...          0
5        5  [paid_search, paid_search, display, affiliate,...          0
6        6                               [affiliate, display]          0
7        7                                        [affiliate]          0
8        8  [paid_social, display, display, paid_search, a...          0
9        9                [paid_social, paid_search, display]          1

Converted journeys: 3005
Non-converted journeys: 6995


In [3]:
# ── Baseline Attribution Models ───────────────────────────────────────────────

converted = journeys[journeys['converted'] == 1].copy()
channels = ['paid_search', 'paid_social', 'display', 'affiliate', 'organic']

last_touch  = defaultdict(float)
first_touch = defaultdict(float)
linear      = defaultdict(float)

for _, row in converted.iterrows():
    path = row['path']
    
    # Last Touch — 100% credit to last channel
    last_touch[path[-1]] += 1
    
    # First Touch — 100% credit to first channel
    first_touch[path[0]] += 1
    
    # Linear — equal credit split across all channels in path
    credit = 1.0 / len(path)
    for ch in path:
        linear[ch] += credit

# Normalize to percentages
total = converted.shape[0]

df_baseline = pd.DataFrame({
    'channel': channels,
    'last_touch':  [last_touch[ch]  / total * 100 for ch in channels],
    'first_touch': [first_touch[ch] / total * 100 for ch in channels],
    'linear':      [linear[ch]      / total * 100 for ch in channels],
})

print(df_baseline.round(2))

       channel  last_touch  first_touch  linear
0  paid_search       34.44        35.44   35.16
1  paid_social       25.76        25.22   26.17
2      display       21.33        21.23   20.98
3    affiliate       10.78        11.11   10.69
4      organic        7.69         6.99    6.99
